In [ ]:
import os 
from typing import Annotated 
from typing_extensions import TypedDict
from dotenv import load_dotenv 
from langchain_groq import ChatGroq 
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph,START,END 
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition #prebuilt router


In [2]:
load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is missing ")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY is missing ")

In [3]:
class State(TypedDict):
    messages:Annotated[list,add_messages]

In [4]:
travel_search=TavilySearch(max_results=5)
tools=[travel_search]

In [5]:
model=ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0
)

In [6]:
model_with_tools=model.bind_tools(tools=tools)

In [7]:
def agent(state:State):
    response=model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [8]:
builder=StateGraph(State)
builder.add_node("agent",agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent",tools_condition)
builder.add_edge("tools", "agent")
app=builder.compile()

In [10]:
res=app.invoke({"messages":[{"role": "user", "content": "find me a hotel in New York"}]})
print(res["messages"][-1].content)

New York City offers a vast range of hotels, from historic luxury palaces to modern budget-friendly stays. To help you find the right one, here are top recommendations categorized by style and price point based on recent traveler reviews and rankings:

### 🏆 Top Luxury Hotels
*   **The Ritz-Carlton New York, NoMad:** Consistently ranked #1 by Condé Nast Traveler readers. Known for its stunning architecture and exceptional service.
*   **The St. Regis New York:** A classic choice in Midtown, famous for its butler service and historic elegance.
*   **The Mark:** A chic, high-end hotel on Madison Avenue, popular with celebrities and design lovers.
*   **Four Seasons Hotel New York Downtown:** Offers a sophisticated experience with views of the Hudson River and the Statue of Liberty.

### 🏙️ Best Mid-Range & Popular Picks
*   **YOTEL New York Times Square:** A modern, tech-forward hotel right in the heart of the action. Great for solo travelers or couples who want a central location.
*   *